# Prepare Documentation
## Materiale finale per il Technical Analysis Document

1. Stampa l'**outline** del documento tecnico (7 sezioni, ~10 pagine)
2. Genera **`RESULTS_SUMMARY.md`** con metriche reali, key findings, raccomandazioni
3. Esporta **`export_for_report.json`** con tutte le metriche unificate

In [1]:
# ── Cella 1: Outline del Technical Analysis Document ──────────────
outline = """
═══════════════════════════════════════════════════════════════════
 TECHNICAL ANALYSIS DOCUMENT — OUTLINE (~10 pagine)
 Plant Disease Detection: Comparative Analysis of 4 Approaches
═══════════════════════════════════════════════════════════════════

1. PROBLEM STATEMENT                                       (~1.5 pag)
   1.1 Motivazione (importanza plant disease detection)
   1.2 Dataset Plant Village — 54.000 immagini, 38 classi
   1.3 Obiettivo: confrontare 4 paradigmi su stesso task

2. METHODOLOGY                                             (~3 pag)
   2.1 V1 — HOG features + SVM (RBF kernel)
   2.2 V2 — Custom CNN da zero (4 blocchi Conv+BN+ReLU+Pool)
   2.3 V3 — ResNet50 Transfer Learning (freeze 1-3, fine-tune layer4)
   2.4 V4 — DINOv3 SSL Foundation Model + Linear Probe / k-NN
   2.5 Preprocessing: split 70/15/15 stratified, ImageNet normalization

3. EXPERIMENTAL SETUP                                      (~1 pag)
   3.1 Hardware: Apple Silicon MPS / consumer GPU
   3.2 Hyperparameters per versione (batch, lr, epoch, patience)
   3.3 Metriche: Accuracy, Precision, Recall, F1 (weighted)
   3.4 Codice riproducibile: notebook 00 → 06, seed fissato

4. RESULTS                                                 (~2.5 pag)
   4.1 Tabella metriche su test set (V1 vs V2 vs V3 vs V4)
   4.2 Confusion matrices per ciascuna versione
   4.3 Training curves V2 vs V3 (V1 single-shot, V4 zero-train)
   4.4 Trade-off tempo/parametri/accuracy
   4.5 Discussion: perché TL batte custom CNN; perché SSL frozen è
       sorprendentemente competitivo con zero fine-tuning

5. FAILURE ANALYSIS                                        (~1 pag)
   5.1 Classi più difficili (per-class F1)
   5.2 Pattern di confusione comuni (foglie visivamente simili)
   5.3 Failure modes specifici per paradigma

6. ETHICAL & PRIVACY CONSIDERATIONS                        (~0.5 pag)
   6.1 Bias del dataset (sfondi controllati, geografia limitata)
   6.2 Deployment in agricoltura: chi accede ai dati farm?
   6.3 Costo computazionale e impatto ambientale

7. CONCLUSIONS                                             (~0.5 pag)
   7.1 Best practice: TL (V3) per produzione, V4 per nuovi domini
   7.2 Limiti e direzioni future
═══════════════════════════════════════════════════════════════════
"""
print(outline)


═══════════════════════════════════════════════════════════════════
 TECHNICAL ANALYSIS DOCUMENT — OUTLINE (~10 pagine)
 Plant Disease Detection: Comparative Analysis of 4 Approaches
═══════════════════════════════════════════════════════════════════

1. PROBLEM STATEMENT                                       (~1.5 pag)
   1.1 Motivazione (importanza plant disease detection)
   1.2 Dataset Plant Village — 54.000 immagini, 38 classi
   1.3 Obiettivo: confrontare 4 paradigmi su stesso task

2. METHODOLOGY                                             (~3 pag)
   2.1 V1 — HOG features + SVM (RBF kernel)
   2.2 V2 — Custom CNN da zero (4 blocchi Conv+BN+ReLU+Pool)
   2.3 V3 — ResNet50 Transfer Learning (freeze 1-3, fine-tune layer4)
   2.4 V4 — DINOv3 SSL Foundation Model + Linear Probe / k-NN
   2.5 Preprocessing: split 70/15/15 stratified, ImageNet normalization

3. EXPERIMENTAL SETUP                                      (~1 pag)
   3.1 Hardware: Apple Silicon MPS / consumer GPU
   3.2 Hy

In [2]:
# ── Cella 2: Genera RESULTS_SUMMARY.md con metriche reali ──────────────
import json
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path("..").resolve()
METRICS_DIR  = PROJECT_ROOT / "results" / "metrics"
SUMMARY_PATH = PROJECT_ROOT / "RESULTS_SUMMARY.md"

def load_metrics(path):
    d = json.loads(path.read_text())
    if "test" in d:
        return {
            "accuracy":  d["test"]["accuracy"],
            "precision": d["test"]["precision"],
            "recall":    d["test"]["recall"],
            "f1":        d["test"]["f1"],
            "training_time_min": d["config"]["train_time_s"] / 60,
            "raw": d,
        }
    return {**d, "raw": d}

v1 = load_metrics(METRICS_DIR / "v1_metrics.json")
v2 = load_metrics(METRICS_DIR / "v2_metrics.json")
v3 = load_metrics(METRICS_DIR / "v3_metrics.json")
v4 = load_metrics(METRICS_DIR / "v4_metrics.json")

rows = [
    ("V1 — HOG + SVM",                v1, "Shallow Learning"),
    ("V2 — Custom CNN",               v2, "Deep Learning from scratch"),
    ("V3 — ResNet50 Transfer Learn.", v3, "Supervised TL + fine-tuning"),
    ("V4 — DINOv3 + Linear Probe",    v4, "SSL Foundation Model frozen"),
]

def pct(x): return f"{x*100:.2f}%"

lines = []
lines.append("# Plant Disease Detection — Results Summary")
lines.append("")
lines.append(f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}_")
lines.append("")
lines.append("Dataset: **Plant Village** (~54.000 immagini, 38 classi)  ")
lines.append("Split: 70% train / 15% val / 15% test (stratified, seed=42)")
lines.append("")

# ── Tabella sintetica ────────────────────────────────────
lines.append("## Tabella Metriche (test set)")
lines.append("")
lines.append("| Versione | Approccio | Accuracy | Precision | Recall | F1 |")
lines.append("|----------|-----------|:--------:|:---------:|:------:|:--:|")
for name, m, approach in rows:
    lines.append(f"| {name} | {approach} | {pct(m['accuracy'])} | {pct(m['precision'])} | {pct(m['recall'])} | {pct(m['f1'])} |")
lines.append("")

# ── Key Findings ──────────────────────────────────────
lines.append("## Key Findings")
lines.append("")
best = max(rows, key=lambda r: r[1]["accuracy"])
lines.append(f"- **Migliore overall:** {best[0]} — accuracy {pct(best[1]['accuracy'])}, F1 {pct(best[1]['f1'])}")
gap_v3_v1 = (v3["accuracy"] - v1["accuracy"]) * 100
lines.append(f"- **Gap V1 → V3:** +{gap_v3_v1:.1f} punti di accuracy. La feature extraction manuale (HOG) collassa rispetto a feature apprese end-to-end.")
lines.append(f"- **V4 SSL competitivo senza training:** {pct(v4['accuracy'])} di accuracy con backbone **completamente congelato** (0 parametri trainable nel backbone). Solo k-NN o un logistic regression sopra le feature.")
v2_time = v2["training_time_min"]
v3_time = v3["training_time_min"]
if v3_time < v2_time:
    lines.append(f"- **Transfer learning più veloce di un custom CNN:** V3 converge in {v3.get('raw', v3).get('best_epoch', '?')} epoch (~{v3_time:.0f} min) vs V2 {v2.get('raw', v2).get('best_epoch','?')} epoch (~{v2_time:.0f} min).")
else:
    lines.append(f"- **Convergenza per epoch più rapida con TL:** V3 raggiunge il best a epoch {v3.get('raw', v3).get('best_epoch', '?')}, V2 a epoch {v2.get('raw', v2).get('best_epoch','?')}.")
v4_raw = v4["raw"]
if isinstance(v4_raw, dict) and "knn" in v4_raw and "linear_probe" in v4_raw:
    lines.append(f"- **V4 k-NN vs Linear Probe:** k-NN {pct(v4_raw['knn']['accuracy'])} vs LinProbe {pct(v4_raw['linear_probe']['accuracy'])}. Il linear probe sfrutta meglio la struttura globale dello spazio di feature.")
lines.append("")

# ── Raccomandazioni di deployment ──────────────────────────
lines.append("## Raccomandazioni per il Deployment")
lines.append("")
lines.append("| Scenario | Versione consigliata | Motivazione |")
lines.append("|----------|----------------------|-------------|")
lines.append("| Max accuracy, dataset fisso | **V3 (ResNet50 TL)** | Best score assoluto, parametri ragionevoli (~24M totali). |")
lines.append("| Onboarding rapido di nuove classi | **V4 (DINOv3 + linear probe)** | Backbone congelato, basta riaddestrare il logistic regression in secondi. |")
lines.append("| Edge / CPU-only / interpretabilità | **V1 (HOG+SVM)** | Modello piccolo, ispezionabile, ma accuracy ~75%. |")
lines.append("| Custom architecture per ricerca / didattica | **V2 (CNN from scratch)** | Pieno controllo dell'architettura, ottimo per oral exam. |")
lines.append("")

# ── Note sulla riproducibilità ─────────────────────────────
lines.append("## Riproducibilità")
lines.append("")
lines.append("- Seed fissato (`random_state=42`) su split, sklearn, torch.")
lines.append("- Notebook ordinati: `00_setup_and_data` → `06_prepare_documentation`.")
lines.append("- Dipendenze in `requirements.txt`. V4 richiede `transformers` + login HuggingFace per DINOv3.")
lines.append("- Checkpoints in `results/models/<version>/` (esclusi da git tramite `.gitignore`).")
lines.append("- Cache embedding V4 in `results/models/v4_dinov3_probe/embeddings/*.npz` (rigenerabili in ~13 min).")
lines.append("")

SUMMARY_PATH.write_text("\n".join(lines))
print(f"Salvato: {SUMMARY_PATH} ✅")
print(f"\n── Anteprima primi 30 righe ──\n")
print("\n".join(lines[:30]))

Salvato: /Users/marco/Documents/repos/ComputerVisionProject/RESULTS_SUMMARY.md ✅

── Anteprima primi 30 righe ──

# Plant Disease Detection — Results Summary

_Generated: 2026-05-30 20:17_

Dataset: **Plant Village** (~54.000 immagini, 38 classi)  
Split: 70% train / 15% val / 15% test (stratified, seed=42)

## Tabella Metriche (test set)

| Versione | Approccio | Accuracy | Precision | Recall | F1 |
|----------|-----------|:--------:|:---------:|:------:|:--:|
| V1 — HOG + SVM | Shallow Learning | 74.39% | 74.62% | 74.39% | 74.26% |
| V2 — Custom CNN | Deep Learning from scratch | 99.66% | 99.66% | 99.66% | 99.66% |
| V3 — ResNet50 Transfer Learn. | Supervised TL + fine-tuning | 99.87% | 99.88% | 99.87% | 99.87% |
| V4 — DINOv3 + Linear Probe | SSL Foundation Model frozen | 98.35% | 98.39% | 98.35% | 98.35% |

## Key Findings

- **Migliore overall:** V3 — ResNet50 Transfer Learn. — accuracy 99.87%, F1 99.87%
- **Gap V1 → V3:** +25.5 punti di accuracy. La feature extraction manuale (HO

In [3]:
# ── Cella 3: Esporta export_for_report.json (metriche unificate) ──────
from datetime import datetime

export = {
    "meta": {
        "project":   "Plant Disease Detection",
        "generated": datetime.now().isoformat(timespec="seconds"),
        "dataset":   "Plant Village (54.000 immagini, 38 classi)",
        "split":     "70/15/15 stratified, seed=42",
    },
    "versions": {},
}

for ver, (name, m, approach) in zip(["v1", "v2", "v3", "v4"], rows):
    raw = m["raw"]
    entry = {
        "display_name": name,
        "approach":     approach,
        "metrics": {
            "accuracy":  round(m["accuracy"], 4),
            "precision": round(m["precision"], 4),
            "recall":    round(m["recall"], 4),
            "f1":        round(m["f1"], 4),
        },
    }
    if ver == "v1":
        entry["training_time_min"] = round(raw["config"]["train_time_s"] / 60, 2)
        entry["inference_ms_per_image"] = round(raw["test"]["inference_ms_per_image"], 2)
        entry["hyperparams"] = {"kernel": raw["config"]["svm_kernel"], "C": raw["config"]["svm_C"]}
    elif ver == "v2":
        entry["training_time_min"] = raw.get("training_time_min")
        entry["best_epoch"]        = raw.get("best_epoch")
    elif ver == "v3":
        entry["training_time_min"] = raw.get("training_time_min")
        entry["best_epoch"]        = raw.get("best_epoch")
        entry["trainable_params"]  = raw.get("trainable_params")
        entry["total_params"]      = raw.get("total_params")
    elif ver == "v4":
        entry["backbone"]          = raw.get("backbone")
        entry["embedding_dim"]     = raw.get("embedding_dim")
        entry["extract_time_min"]  = raw.get("extract_time_min")
        entry["backbone_params"]   = raw.get("backbone_params")
        entry["knn"]               = raw.get("knn")
        entry["linear_probe"]      = raw.get("linear_probe")
        entry["best_classifier"]   = raw.get("best_classifier")
    export["versions"][ver] = entry

# Ranking
ranked = sorted(export["versions"].items(),
                key=lambda kv: kv[1]["metrics"]["accuracy"], reverse=True)
export["ranking_by_accuracy"] = [
    {"version": k, "name": v["display_name"], "accuracy": v["metrics"]["accuracy"]}
    for k, v in ranked
]

out_path = METRICS_DIR / "export_for_report.json"
out_path.write_text(json.dumps(export, indent=2))
print(f"Salvato: {out_path} ✅")
print("\n── Ranking finale per accuracy ──")
for i, e in enumerate(export["ranking_by_accuracy"], 1):
    print(f"  {i}. {e['name']:40s}  accuracy = {e['accuracy']*100:.2f}%")

print("\nDone! Notebook 06 completato — tutto pronto per il Technical Analysis Document.")

Salvato: /Users/marco/Documents/repos/ComputerVisionProject/results/metrics/export_for_report.json ✅

── Ranking finale per accuracy ──
  1. V3 — ResNet50 Transfer Learn.             accuracy = 99.87%
  2. V2 — Custom CNN                           accuracy = 99.66%
  3. V4 — DINOv3 + Linear Probe                accuracy = 98.35%
  4. V1 — HOG + SVM                            accuracy = 74.39%

Done! Notebook 06 completato — tutto pronto per il Technical Analysis Document.
